# 02 — Designing Custom Exception Hierarchies

Built-in exceptions are great for built-in operations. For *your domain*, you want exception **types you defined**. That lets callers write `except YourLibraryError:` without accidentally catching every `ValueError` in Python.

## The minimum custom exception

Subclass `Exception`. Don't subclass `BaseException`. Don't add `__init__` unless you need extra fields.

In [1]:
class PaymentError(Exception):
    """Anything wrong with processing a payment."""

# That's it. Use it like any other exception:
try:
    raise PaymentError("card expired")
except PaymentError as e:
    print("caught:", e)

caught: card expired


## A small hierarchy — the right way

Have one **base** for your module/package, then specific subclasses. Callers can catch broadly (the base) or narrowly (a subclass) — *their choice*.

In [2]:
class PaymentError(Exception):
    """Base for every error in the payments module."""

class PaymentDeclined(PaymentError):
    """The bank declined the charge (insufficient funds, card expired, ...)."""

class PaymentNetworkError(PaymentError):
    """Couldn't reach the payment processor."""

class FraudSuspected(PaymentDeclined):
    """More specific decline reason; still a decline."""

# A caller who wants to retry only on network issues:
def try_payment(raise_what):
    raise raise_what("simulated")

for kind in [PaymentNetworkError, PaymentDeclined, FraudSuspected]:
    try:
        try_payment(kind)
    except PaymentNetworkError:
        print(f"{kind.__name__}: retry later")
    except PaymentDeclined as e:
        # catches both PaymentDeclined and FraudSuspected
        print(f"{kind.__name__}: tell user — {e}")

PaymentNetworkError: retry later
PaymentDeclined: tell user — simulated
FraudSuspected: tell user — simulated


## When to add fields

Add fields when callers will want to *react* programmatically — not just log the message.

In [3]:
class HTTPError(Exception):
    def __init__(self, status_code: int, url: str, body: str = ""):
        # Build a useful default message, then let `super().__init__` store it on `args`.
        super().__init__(f"HTTP {status_code} for {url}")
        self.status_code = status_code
        self.url = url
        self.body = body

try:
    raise HTTPError(503, "https://api.example.com/v1", body="upstream timeout")
except HTTPError as e:
    print(e)                            # nice message
    print("status:", e.status_code)     # programmatic field
    if 500 <= e.status_code < 600:
        print("  -> retry candidate")

HTTP 503 for https://api.example.com/v1
status: 503
  -> retry candidate


## When NOT to add fields

If the only difference between two errors is the message, **don't** invent two classes. Use one class and put the detail in the message. Hierarchy should reflect *handling*, not *flavor*.

## Inherit from a *built-in* exception when it fits

If your custom error is *fundamentally a kind of* something built-in, subclass that — callers can keep using their existing `except ValueError:` and still narrow further when they need to.

In [4]:
class NegativeAmount(ValueError):
    """A money amount went below zero — still a ValueError, just more specific."""

def withdraw(balance: float, amount: float) -> float:
    if amount < 0:
        raise NegativeAmount(f"amount must be non-negative: {amount}")
    if amount > balance:
        raise ValueError(f"insufficient funds: {amount} > {balance}")
    return balance - amount

# Caller A only knows about ValueError — still works
try:
    withdraw(100, -1)
except ValueError as e:
    print("caller A caught", type(e).__name__, ":", e)

# Caller B wants to handle NegativeAmount specially
try:
    withdraw(100, -1)
except NegativeAmount as e:
    print("caller B handled negative amount specifically:", e)
except ValueError as e:
    print("caller B treated as generic value error:", e)

caller A caught NegativeAmount : amount must be non-negative: -1
caller B handled negative amount specifically: amount must be non-negative: -1


## Translating other libraries' exceptions — the boundary pattern

When your module wraps a third-party library, **convert their exceptions into yours** at the boundary. That way your callers depend on *your* exception API, not on someone else's. Combine with `raise ... from e` to keep the cause.

In [5]:
import json

class ConfigError(Exception):
    """Anything wrong with loading or validating config."""

class ConfigParseError(ConfigError):
    """The config text was not valid JSON."""

class ConfigMissingField(ConfigError):
    """Required field missing from the config."""

def load_config(text: str) -> dict:
    try:
        data = json.loads(text)
    except json.JSONDecodeError as e:
        # Translate the third-party exception into our domain — but keep the cause.
        raise ConfigParseError("config is not valid JSON") from e

    if "host" not in data:
        raise ConfigMissingField("missing required field: 'host'")
    return data

for label, text in [("bad json", "{not json}"), ("missing field", '{"port":1}'), ("ok", '{"host":"x"}')]:
    try:
        print(label, "->", load_config(text))
    except ConfigError as e:
        print(label, "->", type(e).__name__, ":", e)
        if e.__cause__:
            print("   caused by:", repr(e.__cause__))

bad json -> ConfigParseError : config is not valid JSON
   caused by: JSONDecodeError('Expecting property name enclosed in double quotes: line 1 column 2 (char 1)')
missing field -> ConfigMissingField : missing required field: 'host'
ok -> {'host': 'x'}


## ExceptionGroup — multiple at once (3.11+)

When you process several items and some fail, you may want to report **all** failures, not just the first. `ExceptionGroup` collects them; `except*` unpacks them.

In [6]:
def process_all(items):
    errors = []
    results = []
    for x in items:
        try:
            results.append(int(x))
        except ValueError as e:
            errors.append(e)
    if errors:
        raise ExceptionGroup("some items failed to parse", errors)
    return results

try:
    process_all(["1", "bad", "3", "oops"])
except* ValueError as eg:
    for e in eg.exceptions:
        print("  -", e)

  - invalid literal for int() with base 10: 'bad'
  - invalid literal for int() with base 10: 'oops'


## Mini-exercise

1. Design exceptions for a tiny **CSV reader**: `CSVError`, `CSVParseError`, `CSVColumnMissing` (with field `column_name`). Write a `read_csv` that uses them. Demonstrate one caller catching the base, another catching the specific.
2. Take the `BankAccount` from
   [03_oop_basics/03_bank_account/account.py](../03_oop_basics/03_bank_account/account.py).
   It currently raises bare `ValueError`. Replace those with a small hierarchy:
   `AccountError` (base), `InvalidOwner`, `InvalidCurrency`, `InsufficientFunds`,
   `NonPositiveAmount`. Which existing tests still pass without modification? Which need
   updating?
3. Write `parse_int(s: str) -> int` that re-raises `int`'s `ValueError` as your own
   `ParseError`, with the original *cause* preserved. Show the chained traceback by
   letting it propagate to the top.

In [7]:
# your solutions here


**Takeaways**

- Subclass `Exception` (never `BaseException`).
- Have one **base** for your module and *specific* subclasses below it. Callers pick the level they want.
- Subclass a *built-in* exception when your error is fundamentally one — `class NegativeAmount(ValueError)`.
- At module boundaries, **translate** other libraries' exceptions into your own with `raise ... from e`.
- Use `ExceptionGroup` to report multiple failures from one operation.

Next: [03_context_managers.ipynb](03_context_managers.ipynb) — `with` is more general than you think.